In [2]:
"""
scale_is_pytorch.py

Positive-contrast example for importance sampling in neural Bayes estimator training.

Problem
-------
    log sigma ~ Uniform(log sigma_lo, log sigma_hi)
    y_1, ..., y_n | sigma ~ N(0, sigma^2)
    network observes s = sqrt(mean(y_i^2))
    network predicts sigma

The Bayes risk is dominated by large sigma because the squared error scales roughly
like sigma^2. Under the log-uniform prior, p(sigma) ∝ 1/sigma, large sigma values are
not sampled often in raw sigma-space.

The gradient-variance heuristic says the useful proposal is

    q*(sigma) ∝ p(sigma) sqrt(L(sigma)).

Since L(sigma) roughly scales like sigma^2, sqrt(L) roughly scales like sigma, so

    q*(sigma) ∝ (1/sigma) * sigma = constant.

Therefore the useful proposal is approximately uniform in sigma.

This script compares:
    1. prior sampling:       sigma ~ p(sigma) ∝ 1/sigma
    2. IS proposal:          sigma ~ Uniform(sigma_lo, sigma_hi)
    3. over-tilted proposal: q(sigma) ∝ sigma

All curves are evaluated under the original prior.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple
import copy
import math
import numpy as np

import torch
import torch.nn as nn

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# Configuration
# ============================================================

@dataclass
class Cfg:
    # Wider range than [0.1, 10] to accentuate the difference.
    sigma_lo: float = 0.03
    sigma_hi: float = 30.0

    # Data and training.
    n_obs: int = 20
    batch: int = 4
    total_budget: int = 100_000
    eval_every: int = 50

    # Evaluation.
    n_test: int = 20_000
    n_seeds: int = 16

    # Network.
    hidden: int = 32

    # Optimiser.
    lr: float = 0.003
    momentum: float = 0.0
    grad_clip: float = 20.0

    # Training objective.
    # True: stable self-normalised IS.
    # False: raw unnormalised IS, not recommended here unless you include constants carefully.
    use_snis: bool = True

    # Device.
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# Utilities
# ============================================================

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def tensor(x: float, cfg: Cfg) -> torch.Tensor:
    return torch.tensor(x, dtype=torch.float32, device=cfg.device)


# ============================================================
# Model
# ============================================================

class MLP(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

        self.reset_parameters()

    def reset_parameters(self) -> None:
        # Xavier is usually better for tanh networks than He initialisation.
        for module in self.net:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

        # Match the spirit of the NumPy script: start with near-zero predictions.
        final = self.net[-1]
        nn.init.zeros_(final.weight)
        nn.init.zeros_(final.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ============================================================
# Simulation
# ============================================================

def draw_sigma_prior(batch: int, cfg: Cfg) -> torch.Tensor:
    """
    Draw from the log-uniform prior.

        p(sigma) ∝ 1 / sigma
        sigma in [sigma_lo, sigma_hi]
    """
    lo = tensor(cfg.sigma_lo, cfg)
    hi = tensor(cfg.sigma_hi, cfg)

    u = torch.rand(batch, device=cfg.device)

    sigma = torch.exp(torch.log(lo) + u * torch.log(hi / lo))

    return sigma


def draw_sigma_proposal(
    batch: int,
    cfg: Cfg,
    tilt: float,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Draw from proposal

        q(sigma) ∝ sigma^tilt

    on [sigma_lo, sigma_hi].

    The unnormalised importance weight is

        w(sigma) = p(sigma) / q(sigma)
                 ∝ sigma^{-1} / sigma^{tilt}
                 = sigma^{-(1 + tilt)}.

    Constants cancel in self-normalised IS.
    """
    lo = cfg.sigma_lo
    hi = cfg.sigma_hi

    u = torch.rand(batch, device=cfg.device)

    if abs(tilt) < 1e-12:
        # q(sigma) ∝ 1, uniform in sigma.
        sigma = lo + (hi - lo) * u
    else:
        # Inverse-CDF for q(sigma) ∝ sigma^tilt.
        power = tilt + 1.0

        if abs(power) < 1e-12:
            # Special case tilt = -1 gives log-uniform, i.e. the prior.
            sigma = draw_sigma_prior(batch, cfg)
        else:
            sigma = (lo ** power + u * (hi ** power - lo ** power)) ** (1.0 / power)

    w = sigma ** (-(1.0 + tilt))

    return sigma, w


def statistic(sigma: torch.Tensor, cfg: Cfg) -> torch.Tensor:
    """
    Simulate y_1, ..., y_n ~ N(0, sigma^2), then return

        s = sqrt(mean(y_i^2)).

    Shape:
        sigma: (B,)
        output: (B, 1)
    """
    y = torch.randn(sigma.shape[0], cfg.n_obs, device=cfg.device) * sigma[:, None]
    s = torch.sqrt(torch.mean(y ** 2, dim=1, keepdim=True))
    return s


def make_test_set(cfg: Cfg, seed: int = 123) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Fixed prior test set used for all methods and all seeds.

    Evaluation is always under the original prior, not under the proposal.
    """
    set_seed(seed)

    sigma = draw_sigma_prior(cfg.n_test, cfg)
    x = statistic(sigma, cfg) / cfg.sigma_hi

    return x, sigma


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model: nn.Module,
    test_x: torch.Tensor,
    test_sigma: torch.Tensor,
    cfg: Cfg,
) -> float:
    model.eval()

    pred_scaled = model(test_x).squeeze(1)
    pred_sigma = pred_scaled * cfg.sigma_hi

    mse = torch.mean((pred_sigma - test_sigma) ** 2)

    return float(mse.item())


# ============================================================
# Training
# ============================================================

def train_run(
    cfg: Cfg,
    seed: int,
    proposal_tilt: Optional[float],
    test_x: torch.Tensor,
    test_sigma: torch.Tensor,
    init_state: Dict[str, torch.Tensor],
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    """
    Train one run.

    proposal_tilt = None:
        draw sigma from the prior.

    proposal_tilt = 0:
        draw sigma uniformly in sigma. This is the predicted good proposal.

    proposal_tilt = 1:
        draw from q(sigma) ∝ sigma. This deliberately over-samples large sigma.
    """
    set_seed(seed)

    model = MLP(cfg.hidden).to(cfg.device)
    model.load_state_dict(copy.deepcopy(init_state))

    opt = torch.optim.SGD(
        model.parameters(),
        lr=cfg.lr,
        momentum=cfg.momentum,
    )

    nsteps = cfg.total_budget // cfg.batch

    budgets: List[int] = []
    risks: List[float] = []

    n_clipped = 0
    grad_norm_sum = 0.0
    grad_norm_max = 0.0

    for step in range(nsteps):
        model.train()

        if proposal_tilt is None:
            sigma = draw_sigma_prior(cfg.batch, cfg)
            w = None
        else:
            sigma, w = draw_sigma_proposal(cfg.batch, cfg, proposal_tilt)

        x = statistic(sigma, cfg) / cfg.sigma_hi
        target = sigma[:, None] / cfg.sigma_hi

        pred = model(x)

        per_loss = (pred - target).squeeze(1) ** 2

        if proposal_tilt is None:
            loss = per_loss.mean()
        else:
            if cfg.use_snis:
                # Self-normalised importance sampling:
                #
                #     loss = sum_j normalized_w_j L_j
                #
                # This is stable and keeps the loss scale similar to the prior case.
                wn = w / (w.sum() + 1e-12)
                loss = torch.sum(wn * per_loss)
            else:
                # Raw IS without normalising constants.
                # Usually less stable. Included only for experimentation.
                loss = torch.mean(w * per_loss)

        opt.zero_grad(set_to_none=True)
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=cfg.grad_clip,
        )

        grad_norm_value = float(grad_norm.item())
        grad_norm_sum += grad_norm_value
        grad_norm_max = max(grad_norm_max, grad_norm_value)

        if grad_norm_value > cfg.grad_clip:
            n_clipped += 1

        opt.step()

        if step % cfg.eval_every == 0 or step == nsteps - 1:
            risk = evaluate(model, test_x, test_sigma, cfg)
            budgets.append((step + 1) * cfg.batch)
            risks.append(risk)

    diagnostics = {
        "clip_fraction": n_clipped / nsteps,
        "mean_grad_norm": grad_norm_sum / nsteps,
        "max_grad_norm": grad_norm_max,
    }

    return np.array(budgets), np.array(risks), diagnostics


# ============================================================
# Approximate variance ceiling diagnostic
# ============================================================

def estimate_ceiling(cfg: Cfg, seed: int = 999, n_bins: int = 32) -> float:
    """
    Approximate the gradient-variance speedup ceiling using the scalar loss profile.

    The ideal proposal for scalar integrands is

        q*(sigma) ∝ p(sigma) sqrt(L(sigma)).

    A crude speedup ceiling is

        E_p[L] / (E_p[sqrt(L)])^2.

    This is not an exact neural-network gradient theorem; it is a useful diagnostic
    for whether the loss contributions are heterogeneous enough for IS to matter.
    """
    set_seed(seed)

    model = MLP(cfg.hidden).to(cfg.device)
    opt = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum)

    log_edges = torch.linspace(
        math.log(cfg.sigma_lo),
        math.log(cfg.sigma_hi),
        n_bins + 1,
        device=cfg.device,
    )

    loss_sum = torch.zeros(n_bins, device=cfg.device)
    counts = torch.zeros(n_bins, device=cfg.device)

    nsteps = (cfg.total_budget // 2) // cfg.batch

    for _ in range(nsteps):
        sigma = draw_sigma_prior(cfg.batch, cfg)

        x = statistic(sigma, cfg) / cfg.sigma_hi
        target = sigma[:, None] / cfg.sigma_hi

        pred = model(x)
        per_loss = (pred - target).squeeze(1) ** 2

        with torch.no_grad():
            b = torch.bucketize(torch.log(sigma), log_edges) - 1
            b = torch.clamp(b, 0, n_bins - 1)

            loss_sum.scatter_add_(0, b, per_loss.detach())
            counts.scatter_add_(0, b, torch.ones_like(per_loss))

        loss = per_loss.mean()

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

    valid = counts > 0
    ell = loss_sum[valid] / counts[valid]

    ratio = (torch.mean(torch.sqrt(ell)) ** 2) / torch.mean(ell)
    ceiling = 1.0 / ratio

    return float(ceiling.item())


# ============================================================
# Plotting
# ============================================================

def plot_curves(
    curves: Dict[str, Tuple[np.ndarray, np.ndarray]],
    output_path: str,
) -> None:
    """
    curves[name] = (budgets, risks)

    where risks has shape (n_seeds, n_eval_points).
    """
    fig, ax = plt.subplots(figsize=(7.5, 4.8))

    for name, (B, R) in curves.items():
        mean = R.mean(axis=0)
        lo = np.percentile(R, 10, axis=0)
        hi = np.percentile(R, 90, axis=0)

        ax.plot(B, mean, lw=2.0, label=name)
        ax.fill_between(B, lo, hi, alpha=0.15)

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel("simulator calls")
    ax.set_ylabel(r"test MSE in raw $\sigma$")
    ax.set_title("1-D scale estimation: prior sampling vs importance sampling")

    ax.grid(alpha=0.3, which="both")
    ax.legend()

    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


# ============================================================
# Main experiment
# ============================================================

def main() -> None:
    cfg = Cfg()

    print()
    print("1-D scale-estimation experiment")
    print("--------------------------------")
    print(f"device:        {cfg.device}")
    print(f"sigma range:   [{cfg.sigma_lo}, {cfg.sigma_hi}]")
    print(f"n_obs:         {cfg.n_obs}")
    print(f"batch:         {cfg.batch}")
    print(f"budget/run:    {cfg.total_budget:,} simulated datasets")
    print(f"seeds:         {cfg.n_seeds}")
    print(f"hidden:        {cfg.hidden}")
    print(f"lr:            {cfg.lr}")
    print(f"grad_clip:     {cfg.grad_clip}")
    print(f"SNIS:          {cfg.use_snis}")
    print()

    test_x, test_sigma = make_test_set(cfg, seed=123)

    proposal_specs = [
        {
            "key": "prior baseline",
            "tilt": None,
        },
        {
            "key": r"IS: uniform $\sigma$",
            "tilt": 0.0,
        },
        {
            "key": r"over-tilted: $q(\sigma)\propto\sigma$",
            "tilt": 1.0,
        },
    ]

    all_risks: Dict[str, List[np.ndarray]] = {
        spec["key"]: [] for spec in proposal_specs
    }

    diagnostics: Dict[str, List[Dict[str, float]]] = {
        spec["key"]: [] for spec in proposal_specs
    }

    budgets_ref: Optional[np.ndarray] = None

    for sd in range(cfg.n_seeds):
        # Same initial network for all methods within a seed.
        set_seed(10_000 + sd)
        base_model = MLP(cfg.hidden).to(cfg.device)
        init_state = copy.deepcopy(base_model.state_dict())

        print(f"seed {sd + 1:02d}/{cfg.n_seeds}")

        for j, spec in enumerate(proposal_specs):
            key = spec["key"]
            tilt = spec["tilt"]

            budgets, risks, diag = train_run(
                cfg=cfg,
                seed=100_000 + 1000 * sd + j,
                proposal_tilt=tilt,
                test_x=test_x,
                test_sigma=test_sigma,
                init_state=init_state,
            )

            if budgets_ref is None:
                budgets_ref = budgets

            all_risks[key].append(risks)
            diagnostics[key].append(diag)

            print(
                f"  {key:<34s} "
                f"final MSE = {risks[-1]:.6f} | "
                f"clip frac = {diag['clip_fraction']:.3f}"
            )

    assert budgets_ref is not None

    curves: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}

    print()
    print("Summary")
    print("-------")

    for spec in proposal_specs:
        key = spec["key"]
        R = np.stack(all_risks[key], axis=0)
        curves[key] = (budgets_ref, R)

        clip_fracs = np.array([d["clip_fraction"] for d in diagnostics[key]])
        mean_grad_norms = np.array([d["mean_grad_norm"] for d in diagnostics[key]])
        max_grad_norms = np.array([d["max_grad_norm"] for d in diagnostics[key]])

        print(f"{key}")
        print(f"  final MSE mean:      {R[:, -1].mean():.6f}")
        print(f"  final MSE std:       {R[:, -1].std(ddof=1):.6f}")
        print(f"  clip fraction mean:  {clip_fracs.mean():.3f}")
        print(f"  mean grad norm:      {mean_grad_norms.mean():.3f}")
        print(f"  max grad norm mean:  {max_grad_norms.mean():.3f}")
        print()

    prior_final = curves["prior baseline"][1][:, -1].mean()
    is_final = curves[r"IS: uniform $\sigma$"][1][:, -1].mean()

    print(
        "Risk reduction of IS relative to prior at final equal budget: "
        f"{100.0 * (prior_final - is_final) / prior_final:.2f}%"
    )

    ceiling = estimate_ceiling(cfg)
    print(f"Approximate scalar-loss speedup ceiling: {ceiling:.2f}x")

    output_path = "scale_is_pytorch_result.png"
    plot_curves(curves, output_path)

    print()
    print(f"saved {output_path}")


if __name__ == "__main__":
    main()


1-D scale-estimation experiment
--------------------------------
device:        cpu
sigma range:   [0.03, 30.0]
n_obs:         20
batch:         4
budget/run:    100,000 simulated datasets
seeds:         16
hidden:        32
lr:            0.003
grad_clip:     20.0
SNIS:          True

seed 01/16
  prior baseline                     final MSE = 1.393891 | clip frac = 0.000
  IS: uniform $\sigma$               final MSE = 1.407961 | clip frac = 0.000
  over-tilted: $q(\sigma)\propto\sigma$ final MSE = 1.833575 | clip frac = 0.000
seed 02/16
  prior baseline                     final MSE = 1.371647 | clip frac = 0.000
  IS: uniform $\sigma$               final MSE = 1.369335 | clip frac = 0.000
  over-tilted: $q(\sigma)\propto\sigma$ final MSE = 1.657688 | clip frac = 0.000
seed 03/16
  prior baseline                     final MSE = 1.361615 | clip frac = 0.000
  IS: uniform $\sigma$               final MSE = 1.375212 | clip frac = 0.000
  over-tilted: $q(\sigma)\propto\sigma$ final MSE